# Chapter 13: Fraud Detection, Compliance, and Financial Crime

This notebook reproduces every worked numerical example in Chapter 13, computed here first and then transcribed into the chapter text, following this book's compute-first discipline.

1. Scaling Chapter 1's Bayes' theorem fraud example to a full day of transaction volume.
2. A second Bayesian application: insider trading surveillance.
3. Fitting a fraud-scoring logistic regression on ten transactions.
4. The precision-recall trade-off at three classification thresholds.
5. Cost-sensitive learning: class-weighted logistic regression.
5b. Resampling: SMOTE synthetic fraud points.
5c. Gradient boosting vs. logistic regression on a genuine feature interaction.
5d. Unsupervised anomaly detection: Isolation Forest without any fraud labels.
6. A Kaplan-Meier survival analysis of flagged accounts.
7. An AML structuring (smurfing) example.
8. A four-hop layering network example.
9. Alert-to-SAR conversion economics.

## 1. Bayes' theorem fraud example at full transaction volume

Chapter 1 assumed a 1% fraud rate, a 95% true-positive rate, and a 3% false-positive rate, giving a posterior $\Pr(\text{Fraud}\mid\text{Flag}) \approx 24.2\%$. Here we scale that to a bank processing 500,000 transactions in a single day.

In [1]:
N = 500_000
fraud_rate = 0.01
tpr = 0.95
fpr = 0.03

n_fraud = N * fraud_rate
n_legit = N * (1 - fraud_rate)

TP = n_fraud * tpr
FN = n_fraud * (1 - tpr)
FP = n_legit * fpr
TN = n_legit * (1 - fpr)

flagged = TP + FP
precision = TP / flagged
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

print(f"n_fraud = {n_fraud:,.0f}, n_legit = {n_legit:,.0f}")
print(f"TP = {TP:,.0f}, FN = {FN:,.0f}, FP = {FP:,.0f}, TN = {TN:,.0f}")
print(f"Flagged transactions = {flagged:,.0f}")
print(f"Precision = {precision:.4f}  (matches Chapter 1's ~24.2% posterior)")
print(f"Recall = {recall:.4f}")
print(f"F1 = {f1:.4f}")

n_fraud = 5,000, n_legit = 495,000
TP = 4,750, FN = 250, FP = 14,850, TN = 480,150
Flagged transactions = 19,600
Precision = 0.2423  (matches Chapter 1's ~24.2% posterior)
Recall = 0.9500
F1 = 0.3862


## 2. A second Bayesian application: insider trading surveillance

A volume-anomaly detector flags 75% of genuine insider-trading episodes (a 2% base rate among reviewed announcements) and false-flags 8% of announcements with no insider trading.

In [2]:
p_insider = 0.02
tpr_insider = 0.75
fpr_insider = 0.08

p_flag = tpr_insider * p_insider + fpr_insider * (1 - p_insider)
p_insider_given_flag = tpr_insider * p_insider / p_flag

print(f"Pr(Flag) = {p_flag:.4f}")
print(f"Pr(Insider | Flag) = {p_insider_given_flag:.4f} = {p_insider_given_flag*100:.1f}%")
print("Compare to the transaction-fraud example's 24.2% precision at a 3% false-positive rate:")
print("a higher false-positive rate (8% vs 3%) dominates a lower true-positive rate (75% vs 95%).")

Pr(Flag) = 0.0934
Pr(Insider | Flag) = 0.1606 = 16.1%
Compare to the transaction-fraud example's 24.2% precision at a 3% false-positive rate:
a higher false-positive rate (8% vs 3%) dominates a lower true-positive rate (75% vs 95%).


## 3. Fraud scoring with logistic regression

Ten transactions, each with three features (amount $z$-score, distance from home in hundreds of km, a late-night flag), used to fit a logistic regression fraud-scoring model, in the same spirit as Chapter 6's confusion matrix example and Chapter 9's credit-scoring model.

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# columns: amount z-score, distance (100s km), late-night flag
X = np.array([
    [3.1, 8.0, 1],   # 1
    [0.2, 0.1, 0],   # 2
    [2.8, 6.5, 1],   # 3
    [1.9, 6.0, 0],   # 4
    [0.4, 0.2, 0],   # 5
    [3.4, 9.2, 1],   # 6
    [0.1, 0.4, 0],   # 7
    [0.6, 0.1, 1],   # 8
    [1.2, 0.3, 1],   # 9
    [0.3, 0.2, 0],   # 10
])
y = np.array([1, 0, 1, 0, 0, 1, 0, 0, 1, 0])

clf = LogisticRegression()
clf.fit(X, y)

print("Coefficients (amount z, distance, late-night):", np.round(clf.coef_[0], 3))
print("Intercept:", np.round(clf.intercept_[0], 3))

proba = clf.predict_proba(X)[:, 1]
pred = (proba >= 0.5).astype(int)

for i, (p, pr, act) in enumerate(zip(proba, pred, y), start=1):
    print(f"Transaction {i}: predicted probability = {p:.3f}, predicted = {pr}, actual = {act}")

Coefficients (amount z, distance, late-night): [0.832 0.146 0.825]
Intercept: -2.575
Transaction 1: predicted probability = 0.880, predicted = 1, actual = 1
Transaction 2: predicted probability = 0.084, predicted = 0, actual = 0
Transaction 3: predicted probability = 0.822, predicted = 1, actual = 1
Transaction 4: predicted probability = 0.470, predicted = 0, actual = 0
Transaction 5: predicted probability = 0.099, predicted = 0, actual = 0
Transaction 6: predicted probability = 0.918, predicted = 1, actual = 1
Transaction 7: predicted probability = 0.081, predicted = 0, actual = 0
Transaction 8: predicted probability = 0.225, predicted = 0, actual = 0
Transaction 9: predicted probability = 0.330, predicted = 0, actual = 1
Transaction 10: predicted probability = 0.091, predicted = 0, actual = 0


In [4]:
TP2 = int(np.sum((pred == 1) & (y == 1)))
FP2 = int(np.sum((pred == 1) & (y == 0)))
FN2 = int(np.sum((pred == 0) & (y == 1)))
TN2 = int(np.sum((pred == 0) & (y == 0)))

precision2 = TP2 / (TP2 + FP2)
recall2 = TP2 / (TP2 + FN2)
f1_2 = 2 * precision2 * recall2 / (precision2 + recall2)

print(f"TP={TP2} FP={FP2} FN={FN2} TN={TN2}")
print(f"Precision = {precision2:.3f}, Recall = {recall2:.3f}, F1 = {f1_2:.3f}")

TP=3 FP=0 FN=1 TN=6
Precision = 1.000, Recall = 0.750, F1 = 0.857


## 4. The precision-recall trade-off at three classification thresholds

Recomputing Section 3's confusion matrix at thresholds of 0.30, 0.50, and 0.85 instead of the original 0.5, to show how the threshold choice trades precision against recall.

In [5]:
for thr in [0.30, 0.50, 0.85]:
    pred_t = (proba >= thr).astype(int)
    TP_t = int(np.sum((pred_t == 1) & (y == 1)))
    FP_t = int(np.sum((pred_t == 1) & (y == 0)))
    FN_t = int(np.sum((pred_t == 0) & (y == 1)))
    TN_t = int(np.sum((pred_t == 0) & (y == 0)))
    prec_t = TP_t / (TP_t + FP_t)
    rec_t = TP_t / (TP_t + FN_t)
    f1_t = 2 * prec_t * rec_t / (prec_t + rec_t)
    print(f"threshold={thr:.2f}: TP={TP_t} FP={FP_t} FN={FN_t} TN={TN_t}  "
          f"precision={prec_t:.3f} recall={rec_t:.3f} F1={f1_t:.3f}")

threshold=0.30: TP=4 FP=1 FN=0 TN=5  precision=0.800 recall=1.000 F1=0.889
threshold=0.50: TP=3 FP=0 FN=1 TN=6  precision=1.000 recall=0.750 F1=0.857
threshold=0.85: TP=2 FP=0 FN=2 TN=6  precision=1.000 recall=0.500 F1=0.667


## 5. Cost-sensitive learning: class-weighted logistic regression

Refitting Section 3's logistic regression with `class_weight='balanced'`, so that fraud (the minority class) is weighted more heavily during training.

In [6]:
clf_w = LogisticRegression(class_weight='balanced')
clf_w.fit(X, y)

print("Weighted coefficients (amount z, distance, late-night):", np.round(clf_w.coef_[0], 3))
print("Weighted intercept:", np.round(clf_w.intercept_[0], 3))

proba_w = clf_w.predict_proba(X)[:, 1]
for i, (p, act) in enumerate(zip(proba_w, y), start=1):
    print(f"Transaction {i}: predicted probability = {p:.3f}, actual = {act}")

Weighted coefficients (amount z, distance, late-night): [0.861 0.13  0.845]
Weighted intercept: -2.254
Transaction 1: predicted probability = 0.909, actual = 1
Transaction 2: predicted probability = 0.112, actual = 0
Transaction 3: predicted probability = 0.864, actual = 1
Transaction 4: predicted probability = 0.541, actual = 0
Transaction 5: predicted probability = 0.132, actual = 0
Transaction 6: predicted probability = 0.938, actual = 1
Transaction 7: predicted probability = 0.108, actual = 0
Transaction 8: predicted probability = 0.293, actual = 0
Transaction 9: predicted probability = 0.417, actual = 1
Transaction 10: predicted probability = 0.122, actual = 0


In [7]:
for thr in [0.50, 0.40]:
    pred_w = (proba_w >= thr).astype(int)
    TP_w = int(np.sum((pred_w == 1) & (y == 1)))
    FP_w = int(np.sum((pred_w == 1) & (y == 0)))
    FN_w = int(np.sum((pred_w == 0) & (y == 1)))
    TN_w = int(np.sum((pred_w == 0) & (y == 0)))
    prec_w = TP_w / (TP_w + FP_w)
    rec_w = TP_w / (TP_w + FN_w)
    f1_w = 2 * prec_w * rec_w / (prec_w + rec_w)
    print(f"threshold={thr:.2f}: TP={TP_w} FP={FP_w} FN={FN_w} TN={TN_w}  "
          f"precision={prec_w:.3f} recall={rec_w:.3f} F1={f1_w:.3f}")

threshold=0.50: TP=3 FP=1 FN=1 TN=5  precision=0.750 recall=0.750 F1=0.750
threshold=0.40: TP=4 FP=1 FN=0 TN=5  precision=0.800 recall=1.000 F1=0.889


## 5b. Resampling: SMOTE synthetic fraud points (Section 13.5)

Applying SMOTE to the same ten-transaction dataset's four fraud cases (transactions 1, 3, 6, 9), asking for two additional synthetic fraud points using $k=3$ nearest neighbors, and verifying each synthetic point lies exactly on the line segment between the two real fraud cases SMOTE interpolated between.

In [8]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(k_neighbors=3, random_state=0)
X_res, y_res = sm.fit_resample(X, y)
synthetic_points = X_res[len(X):]
print("Two synthetic fraud points SMOTE generated:")
print(synthetic_points)

# Verify point A lies on the segment between Transaction 9 (1.2, 0.3) and Transaction 3 (2.8, 6.5)
lam_a = (synthetic_points[0, 0] - 1.2) / (2.8 - 1.2)
y_check_a = 0.3 + lam_a * (6.5 - 0.3)
print(f"\nPoint A: lambda (Txn9->Txn3) = {lam_a:.3f}, predicted y = {y_check_a:.3f}, actual y = {synthetic_points[0,1]:.3f}")

# Verify point B lies on the segment between Transaction 1 (3.1, 8.0) and Transaction 6 (3.4, 9.2)
lam_b = (synthetic_points[1, 0] - 3.1) / (3.4 - 3.1)
y_check_b = 8.0 + lam_b * (9.2 - 8.0)
print(f"Point B: lambda (Txn1->Txn6) = {lam_b:.3f}, predicted y = {y_check_b:.3f}, actual y = {synthetic_points[1,1]:.3f}")

Two synthetic fraud points SMOTE generated:
[[1.8355786  2.76286707 1.        ]
 [3.26346495 8.65385982 1.        ]]

Point A: lambda (Txn9->Txn3) = 0.397, predicted y = 2.763, actual y = 2.763
Point B: lambda (Txn1->Txn6) = 0.545, predicted y = 8.654, actual y = 8.654


## 5c. Gradient boosting vs. logistic regression on a genuine feature interaction (Section 13.5)

The ten-transaction sample is too small to show a real ensemble advantage. Here we simulate 5,000 transactions with a genuine three-way interaction built into the true fraud process: fraud is only sharply more likely when amount, distance, and lateness are *simultaneously* unusual, a rule an additive logistic regression cannot represent no matter how it is fit. Both models are trained with matched class weighting and evaluated on genuinely held-out data using precision-recall AUC, the metric recommended over ROC-AUC under severe imbalance.

In [9]:
from sklearn.ensemble import GradientBoostingClassifier, IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score

rng_sim = np.random.default_rng(5)
n_sim = 5_000

amount_z_sim = rng_sim.gamma(2.0, 1.0, n_sim)
distance_sim = rng_sim.gamma(1.5, 2.0, n_sim)
late_night_sim = rng_sim.binomial(1, 0.15, n_sim)

# True rule: a modest additive baseline, plus a much larger jump when amount,
# distance, AND lateness are all simultaneously unusual (a genuine AND-interaction).
triple_flag = ((amount_z_sim > 2.5) & (distance_sim > 4.0) & (late_night_sim == 1)).astype(float)
base_logit = -4.5 + 0.15*amount_z_sim + 0.05*distance_sim + 0.3*late_night_sim
p_fraud_sim = 1 / (1 + np.exp(-(base_logit + 4.0*triple_flag)))
fraud_sim = rng_sim.binomial(1, p_fraud_sim)
print(f"Simulated {n_sim} transactions; fraud rate: {fraud_sim.mean():.4f} ({fraud_sim.sum()} cases)")

X_sim = np.column_stack([amount_z_sim, distance_sim, late_night_sim])
n_train_sim = 4_000
X_train_s, X_test_s = X_sim[:n_train_sim], X_sim[n_train_sim:]
y_train_s, y_test_s = fraud_sim[:n_train_sim], fraud_sim[n_train_sim:]

logit_sim = LogisticRegression(class_weight="balanced").fit(X_train_s, y_train_s)
pr_auc_logit = average_precision_score(y_test_s, logit_sim.predict_proba(X_test_s)[:, 1])
print(f"\nLogistic regression PR-AUC: {pr_auc_logit:.4f}")

# match logistic regression's balanced class weighting via explicit sample weights
n_pos, n_neg = y_train_s.sum(), len(y_train_s) - y_train_s.sum()
sample_weight = np.where(y_train_s == 1, len(y_train_s)/(2*n_pos), len(y_train_s)/(2*n_neg))
gbm_sim = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, random_state=0)
gbm_sim.fit(X_train_s, y_train_s, sample_weight=sample_weight)
pr_auc_gbm = average_precision_score(y_test_s, gbm_sim.predict_proba(X_test_s)[:, 1])
print(f"Gradient boosting PR-AUC: {pr_auc_gbm:.4f}")
print(f"Improvement: {pr_auc_gbm - pr_auc_logit:.4f} ({(pr_auc_gbm-pr_auc_logit)/pr_auc_logit*100:.1f}% relative)")
print(f"GBM feature importances (amount, distance, late-night): {gbm_sim.feature_importances_}")

Simulated 5000 transactions; fraud rate: 0.0244 (122 cases)

Logistic regression PR-AUC: 0.2847


Gradient boosting PR-AUC: 0.3821
Improvement: 0.0974 (34.2% relative)
GBM feature importances (amount, distance, late-night): [0.40836573 0.49026054 0.10137374]


## 5d. Unsupervised anomaly detection: Isolation Forest without any fraud labels (Section 13.5)

Fitting an Isolation Forest on the same training transactions' features only, with the fraud labels never shown to the model at all, then checking whether its unsupervised anomaly score still separates the held-out fraud cases from legitimate ones.

In [10]:
iso_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=0)
iso_forest.fit(X_train_s)  # unsupervised: y_train_s is never passed in

anomaly_scores = -iso_forest.score_samples(X_test_s)  # higher = more anomalous
mean_score_fraud = anomaly_scores[y_test_s == 1].mean()
mean_score_legit = anomaly_scores[y_test_s == 0].mean()
iso_auc = roc_auc_score(y_test_s, anomaly_scores)

print(f"Mean anomaly score, fraud cases (n={y_test_s.sum()}): {mean_score_fraud:.4f}")
print(f"Mean anomaly score, legitimate cases: {mean_score_legit:.4f}")
print(f"ROC-AUC of the unsupervised anomaly score against held-out labels: {iso_auc:.4f}")
print("(labels used only to evaluate the score after the fact, never during fitting)")

Mean anomaly score, fraud cases (n=34): 0.5525
Mean anomaly score, legitimate cases: 0.4658
ROC-AUC of the unsupervised anomaly score against held-out labels: 0.7428
(labels used only to evaluate the score after the fact, never during fitting)


## 6. Kaplan-Meier survival analysis of flagged accounts

Twelve accounts flagged by a behavioral risk-tiering system, each with an observed time (in days) to either a confirmed fraud closure (event) or a censored observation (unrelated closure, or still open at the end of the observation window).

In [11]:
# (time, event) pairs; event=1 means confirmed-fraud closure, event=0 means censored
data = [
    (5, 1), (8, 1), (8, 0), (12, 1), (15, 1), (15, 0),
    (20, 1), (24, 0), (30, 1), (30, 0), (35, 1), (40, 0),
]

event_times = sorted(set(t for t, e in data if e == 1))

S = 1.0
rows = []
for t in event_times:
    n_i = sum(1 for tt, ee in data if tt >= t)
    d_i = sum(1 for tt, ee in data if tt == t and ee == 1)
    factor = 1 - d_i / n_i
    S *= factor
    rows.append((t, n_i, d_i, factor, S))
    print(f"t={t:>2}  n_i={n_i:>2}  d_i={d_i}  factor={factor:.4f}  S(t)={S:.4f}")

t= 5  n_i=12  d_i=1  factor=0.9167  S(t)=0.9167
t= 8  n_i=11  d_i=1  factor=0.9091  S(t)=0.8333
t=12  n_i= 9  d_i=1  factor=0.8889  S(t)=0.7407
t=15  n_i= 8  d_i=1  factor=0.8750  S(t)=0.6481
t=20  n_i= 6  d_i=1  factor=0.8333  S(t)=0.5401
t=30  n_i= 4  d_i=1  factor=0.7500  S(t)=0.4051
t=35  n_i= 2  d_i=1  factor=0.5000  S(t)=0.2025


## 7. AML structuring (smurfing) example

Five cash deposits of \$9,500 each, individually under the \$10,000 Currency Transaction Report (CTR) threshold.

In [12]:
ctr_threshold = 10_000
deposit = 9_500
n_deposits = 5

total = deposit * n_deposits
print(f"Total deposited across {n_deposits} transactions: ${total:,}")
print(f"Each deposit is ${ctr_threshold - deposit} below the ${ctr_threshold:,} CTR threshold")
print(f"A single ${total:,} deposit would unambiguously trigger a CTR filing")

Total deposited across 5 transactions: $47,500
Each deposit is $500 below the $10,000 CTR threshold
A single $47,500 deposit would unambiguously trigger a CTR filing


## 8. A four-hop layering network example

\$500,000 in placed funds moves through four shell entities, each forwarding 98% of what it received onward within 24 hours.

In [13]:
amount = 500_000
retain_rate = 0.02
entities = ["Alpha", "Beta", "Gamma", "Delta"]

running = amount
total_retained = 0
for name in entities:
    received = running
    retained = received * retain_rate
    forwarded = received - retained
    total_retained += retained
    print(f"{name}: receives ${received:,.2f}, forwards ${forwarded:,.2f}, retains ${retained:,.2f}")
    running = forwarded

print(f"\nFinal amount reaching ultimate destination: ${running:,.2f}")
print(f"Total retained (leakage) across the chain: ${total_retained:,.2f}")
print(f"Percent of original amount reaching the end: {running/amount*100:.2f}%")

Alpha: receives $500,000.00, forwards $490,000.00, retains $10,000.00
Beta: receives $490,000.00, forwards $480,200.00, retains $9,800.00
Gamma: receives $480,200.00, forwards $470,596.00, retains $9,604.00
Delta: receives $470,596.00, forwards $461,184.08, retains $9,411.92

Final amount reaching ultimate destination: $461,184.08
Total retained (leakage) across the chain: $38,815.92
Percent of original amount reaching the end: 92.24%


## 9. Alert-to-SAR conversion economics

A bank's combined fraud and AML monitoring system generates 8,000 alerts per month, of which 240 are filed as Suspicious Activity Reports (SARs).

In [14]:
alerts_per_month = 8_000
sars_filed = 240
minutes_per_alert = 20

conversion_rate = sars_filed / alerts_per_month
analyst_hours = alerts_per_month * minutes_per_alert / 60
fte_analysts = analyst_hours / 160  # ~160 working hours per analyst per month

print(f"Alert-to-SAR conversion rate = {conversion_rate:.4f} = {conversion_rate*100:.2f}%")
print(f"Analyst hours per month to review all alerts = {analyst_hours:,.1f}")
print(f"Equivalent full-time analysts (at 160 hours/month) = {fte_analysts:.1f}")

Alert-to-SAR conversion rate = 0.0300 = 3.00%
Analyst hours per month to review all alerts = 2,666.7
Equivalent full-time analysts (at 160 hours/month) = 16.7
